In [1]:
from kestra_client import KestraClient
import os
from openai import OpenAI
from pathlib import Path

In [2]:
kestra_client = KestraClient()
open_ai_client = OpenAI()

In [3]:
flow_list = [f'flows/{_}' for _ in os.listdir("flows") 
             if _.endswith(".yaml") and _.split("_")[0].isdigit()]
flow_list

['flows/5_web_search_agent.yaml',
 'flows/3_rag_with_websearch.yaml',
 'flows/1_chat_without_rag.yaml',
 'flows/6_multi_agent_research.yaml',
 'flows/2_chat_with_rag.yaml',
 'flows/4_simple_agent.yaml']

In [4]:
for flow in flow_list:
    kestra_client.register_flow(flow)

Flow zoomcamp.5_web_research_agent already exists
Flow zoomcamp.3_rag_with_websearch already exists
Flow zoomcamp.1_chat_without_rag already exists
Flow zoomcamp.6_multi_agent_research already exists
Flow zoomcamp.2_chat_with_rag already exists
Flow zoomcamp.4_simple_agent already exists


**TASK-1:**

Try the following experiment:

Open ChatGPT in a private browser window: https://chatgpt.com

Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"

Then, use Kestra's AI Copilot with the same prompt

In [5]:
prompt="Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"

In [ ]:
#AI generated response
response = open_ai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt)

flow_yaml=response.output_text

output_path = Path('flows')
output_path.mkdir(parents=True, exist_ok=True)

file_path = output_path/'ny_taxi_open_ai.yaml'

file_path.write_text(flow_yaml, encoding="utf-8")

In [ ]:
#Kestra AI Pilot Generated response
kestra_client.generate_flow(user_prompt=prompt,
                            provider_id='gemini-legacy',
                            namespace='zoomcamp',
                            output_dir='flows')

In [ ]:
print("""
      Q-1 : After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?
      A-1 : AI Copilot has access to current Kestra plugin documentation
      """)

**TASK-2**:

Run both 1_chat_without_rag.yaml and 2_chat_with_rag.yaml. Read the execution logs for each.

In [ ]:
execution_1 = kestra_client.execute_flow(flow_id = "1_chat_without_rag",
                                         namespace='zoomcamp')
exec_log_id_1 = execution_1['id']
response = kestra_client.get_execution_logs(execution_id = exec_log_id_1, level='INFO')
kestra_answer_1 = response[0]["message"]
kestra_answer_1

In [ ]:
execution_2 = kestra_client.execute_flow(flow_id = "2_chat_with_rag",
                                         namespace='zoomcamp')
exec_log_id_2 = execution_2['id']
response = kestra_client.get_execution_logs(execution_id = exec_log_id_2, level='INFO')
kestra_answer_2 = response[0]["message"]
kestra_answer_2

In [ ]:
print("""
      Q-2: The non-RAG response about Kestra 1.1 features is best described as:
      A-2: Vague, generic, or fabricated — the model guesses from training data
      """)

**TASK-3:**

Run 4_simple_agent.yaml with summary_length = short (leave the other inputs as defaults).

Open the execution logs and find the token usage logged by the log_token_usage task.

In [21]:
execution_4_short = kestra_client.execute_flow(namespace="zoomcamp",
    flow_id="4_simple_agent",
    inputs={
        "summary_length": "short"}
    )
exec_log_id_4_short = execution_4_short['id']
exec_log_id_4_short

'BjtV4NUQmS8cgpbbGW9iY'

In [ ]:
response = kestra_client.get_execution_logs(execution_id = exec_log_id_4_short, level='INFO')
kestra_answer_4_short = response[0]["message"]
kestra_answer_4_short.split("\n")

['📊 Token Usage Summary:',
 '',
 'Multilingual Agent:',
 '- Input tokens: 282',
 '- Output tokens: 73',
 '- Total tokens: 355',
 '',
 'English Brevity Agent:',
 '- Input tokens: 88',
 '- Output tokens: 35',
 '- Total tokens: 123',
 '',
 '💡 Tip: Monitor token usage to understand costs and optimize prompts!',
 '']

In [25]:
print(f""" 
      Q-3: What is the approximate output token count for multilingual_agent? 
      A-3: {kestra_answer_4_short.split("\n")[4]} """)

 
      Q-3: What is the approximate output token count for multilingual_agent? 
      A-3: - Output tokens: 73 


**TASK - 4:** 

Run 4_simple_agent.yaml again with summary_length = long.

Compare the multilingual_agent output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?

In [41]:
execution_4_long = kestra_client.execute_flow(namespace="zoomcamp",
    flow_id="4_simple_agent",
    inputs={
        "summary_length": "long"}
    )
exec_log_id_4_long = execution_4_long['id']
exec_log_id_4_long

'2R1ffPDk1OluAfm6nCZoDV'

In [43]:
response = kestra_client.get_execution_logs(execution_id = exec_log_id_4_long, level='INFO')
kestra_answer_4_long = response[0]["message"]
kestra_answer_4_long.split("\n")

['📊 Token Usage Summary:',
 '',
 'Multilingual Agent:',
 '- Input tokens: 282',
 '- Output tokens: 174',
 '- Total tokens: 456',
 '',
 'English Brevity Agent:',
 '- Input tokens: 189',
 '- Output tokens: 59',
 '- Total tokens: 248',
 '',
 '💡 Tip: Monitor token usage to understand costs and optimize prompts!',
 '']

In [44]:
print(f""" 
      Q-4: What is the approximate output token count for multilingual_agent? 
      A-4: {kestra_answer_4_long.split("\n")[4]} """)

 
      Q-4: What is the approximate output token count for multilingual_agent? 
      A-4: - Output tokens: 174 


**TASK-5:**

Open 4_simple_agent.yaml

Find the english_brevity task and change its prompt from asking for exactly 1 sentence to asking for exactly 3 sentences.

Save the flow as 4_simple_agent_eng_brevity_3, then run it with summary_length = long.

In [32]:
kestra_client.register_flow('flows/4_simple_agent_eng_brevity_3.yaml')

Registered zoomcamp.4_simple_agent_eng_brevity_3


In [33]:
execution_4_brevity_3 = kestra_client.execute_flow(namespace="zoomcamp",
    flow_id="4_simple_agent_eng_brevity_3",
    inputs={
        "summary_length": "long"}
    )
exec_log_id_4_brevity_3 = execution_4_brevity_3['id']
exec_log_id_4_brevity_3

'2t98Rspya4agnRi9soR1vM'

In [34]:
response = kestra_client.get_execution_logs(execution_id = exec_log_id_4_brevity_3, level='INFO')
kestra_answer_4_brevity_3 = response[0]["message"]
kestra_answer_4_brevity_3.split("\n")

['📊 Token Usage Summary:',
 '',
 'Multilingual Agent:',
 '- Input tokens: 282',
 '- Output tokens: 191',
 '- Total tokens: 473',
 '',
 'English Brevity Agent:',
 '- Input tokens: 206',
 '- Output tokens: 92',
 '- Total tokens: 298',
 '',
 '💡 Tip: Monitor token usage to understand costs and optimize prompts!',
 '']

In [46]:
print(f""" 
      Q-5: Compare the english_brevity output token count to the original 1-sentence version (also with summary_length = long). How do they compare?
      A-5: English Brevity 3 sentences -{kestra_answer_4_brevity_3.split("\n")[9]} Vs English Brevity 1 sentence -{kestra_answer_4_long.split("\n")[9]}""")

 
      Q-5: Compare the english_brevity output token count to the original 1-sentence version (also with summary_length = long). How do they compare?
      A-5: English Brevity 3 sentences -- Output tokens: 92 Vs English Brevity 1 sentence -- Output tokens: 59


**TASK-6:**

Best Practices

In [47]:
print(f"""
      Q-6: Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?
      A-6: Use traditional task-based workflows for predictability and auditability
      """)


      Q-6: Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?
      A-6: Use traditional task-based workflows for predictability and auditability
      
